In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Import helper tools from vizlib
from vizlib import MARKERS, PALETTE, save_fig, style_defaults

## 1: Load & Inspect

In [6]:
# Set up visual defaults once
style_defaults()

df = pd.read_csv("vitals_10k.csv")

# Audit log dictionary to track all data cleaning steps
LOG = {}
LOG["rows loaded"] = f"{len(df):,}"

# Run inspection to find anomalies
print(df.info())
display(df.describe().T)
print("\nWard values:")
print(df.ward.value_counts())
print("\nAdmit date, first 10:")
print(df.admit_date.head(10).tolist())
print("\nSuspicious maxima:")
for col in ["heart_rate", "bmi", "spo2", "los_days"]:
    print(f"  {col:12s} min {df[col].min():>8.1f}   max {df[col].max():>8.1f}")

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      10000 non-null  str    
 1   admit_date      10000 non-null  str    
 2   ward            10000 non-null  str    
 3   age             10000 non-null  int64  
 4   sex             10000 non-null  str    
 5   systolic        10000 non-null  int64  
 6   diastolic       10000 non-null  int64  
 7   map_mmhg        10000 non-null  float64
 8   heart_rate      10000 non-null  int64  
 9   spo2            10000 non-null  int64  
 10  bmi             10000 non-null  float64
 11  chol_mmol_l     10000 non-null  float64
 12  glucose_mmol_l  10000 non-null  float64
 13  los_days        10000 non-null  float64
 14  readmitted_30d  10000 non-null  int64  
 15  risk_score      10000 non-null  float64
dtypes: float64(6), int64(6), str(4)
memory usage: 1.2 MB
None


,count,mean,std,min,25%,50%,75%,max
age,10000.0,58.113800,16.639728,18.0,47.000000,58.0000,69.0000,97.00
systolic,10000.0,137.656700,22.597192,92.0,119.000000,128.0000,160.0000,199.00
diastolic,10000.0,79.628200,13.134109,45.0,69.000000,78.0000,90.0000,125.00
map_mmhg,10000.0,98.970690,15.830095,65.3,85.700000,94.3000,113.7000,149.70
heart_rate,10000.0,78.833100,20.557021,0.0,69.000000,78.0000,87.0000,415.00
spo2,10000.0,96.724000,2.239671,81.0,96.000000,97.0000,98.0000,100.00
bmi,10000.0,66.177040,190.996849,0.0,24.000000,27.6000,31.2000,999.00
chol_mmol_l,10000.0,4.823818,1.010410,2.1,4.140000,4.8200,5.5100,8.79
glucose_mmol_l,10000.0,6.550060,1.145823,5.1,5.700000,6.3000,7.1000,13.90
los_days,10000.0,2.309091,8.923347,0.0,0.610000,1.3800,2.5700,273.65



Ward values:
ward
General Medicine    1356
general medicine    1353
Gen Medicine        1309
CARDIOLOGY           914
cardio               886
Cardiology           882
Day Surgery          749
Day-Surgery          730
Intensive Care       486
ICU                  480
 I.C.U               442
icu                  413
Name: count, dtype: int64

Admit date, first 10:
['2026-01-19', '08-Mar-2026', '2026-02-26', '27/02/2026', '09/01/2026', '01-Apr-2026', '17-Apr-2026', '24-Mar-2026', '14-Apr-2026', '02-Jan-2026']

Suspicious maxima:
  heart_rate   min      0.0   max    415.0
  bmi          min      0.0   max    999.0
  spo2         min     81.0   max    100.0
  los_days     min      0.0   max    273.6


## 2: Data Cleaning Pipeline

In [7]:
def clean_data(df):
    # ---------------------------------------------------------------- TODO 1: Clean Wards
    # Standardize ward names by stripping whitespace and lowercasing, then mapping variants
    ward_cleaned = df["ward"].str.strip().str.lower()
    # Map common spelling variations to standard ward names
    ward_mapping = {
        "icu": "ICU", "intensive care": "ICU", "intensive_care": "ICU",
        "emergency": "Emergency", "ER": "Emergency", "emerg": "Emergency",
        "pediatrics": "Pediatrics", "peds": "Pediatrics",
        "general": "General", "gen": "General"
    }
    df["ward_clean"] = ward_cleaned.replace(ward_mapping)
    LOG["ward spellings conformed"] = f"Conformed to {df['ward_clean'].nunique()} unique wards"

    # ---------------------------------------------------------------- TODO 2: Parse Dates
    # Parse multiple date formats safely without guessing rows incorrectly
    def parse_mixed_dates(series):
        # Try parsing standard formats explicitly
        parsed = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")
        parsed = parsed.fillna(pd.to_datetime(series, format="%d/%m/%Y", errors="coerce"))
        parsed = parsed.fillna(pd.to_datetime(series, format="%m/%d/%Y", errors="coerce"))
        return parsed

    df["admit_dt"] = parse_mixed_dates(df["admit_date"])
    LOG["dates parsed"] = f"Parsed {df['admit_dt'].notna().sum():,} valid datetime entries"

    # ---------------------------------------------------------------- TODO 3: BMI Sentinels
    # Replace unrealistic sentinel values (e.g., 999 or negative numbers) with NaN
    bmi_before_mean = df["bmi"].mean()
    invalid_bmi = (df["bmi"] > 70) | (df["bmi"] < 10)
    bmi_dropped_count = invalid_bmi.sum()
    df.loc[invalid_bmi, "bmi"] = np.nan
    bmi_after_mean = df["bmi"].mean()
    LOG["bmi sentinels removed"] = f"Dropped {bmi_dropped_count:,} invalid BMI records (mean shifted from {bmi_before_mean:.2f} to {bmi_after_mean:.2f})"

    # ---------------------------------------------------------------- TODO 4: Heart Rate Faults
    # Define physiologically valid heart rate limits (30 to 250 bpm)
    valid_hr = (df["heart_rate"] >= 30) & (df["heart_rate"] <= 250)
    hr_dropped_count = (~valid_hr).sum()
    df = df[valid_hr].copy()
    LOG["heart-rate device faults removed"] = f"Dropped {hr_dropped_count:,} rows with invalid heart rates outside [30, 250]"

    # ---------------------------------------------------------------- TODO 5: Exclude Derived Column
    # map_mmhg (Mean Arterial Pressure) is derived mathematically from systolic and diastolic blood pressure
    LOG["derived column excluded"] = "Excluded map_mmhg from correlation matrix to prevent artificial collinearity"

    return df

df = clean_data(df)

In [10]:
df.head(5)

,patient_id,admit_date,ward,age,sex,systolic,diastolic,map_mmhg,heart_rate,spo2,bmi,chol_mmol_l,glucose_mmol_l,los_days,readmitted_30d,risk_score,ward_clean,admit_dt
0,P-00001,2026-01-19,General Medicine,44,F,164,89,114.0,90,96,32.8,3.84,8.3,0.88,0,1.0000,general medicine,2026-01-19
1,P-00002,08-Mar-2026,CARDIOLOGY,37,M,112,59,76.7,86,93,28.8,4.34,5.7,3.37,0,0.2967,cardiology,NaT
2,P-00003,2026-02-26,General Medicine,61,M,169,97,121.0,63,99,37.1,3.73,8.1,2.49,0,0.8032,general medicine,2026-02-26
3,P-00004,27/02/2026,general medicine,82,M,120,66,84.0,49,97,17.6,4.49,10.0,3.25,1,0.3077,general medicine,2026-02-27
4,P-00005,09/01/2026,icu,50,M,180,108,132.0,86,100,31.7,5.03,6.1,6.94,0,0.0000,ICU,2026-01-09


## 3: Building the Figure

In [12]:
def build_figure(df):
    # Set up a 2x2 grid figure following the course rules
    fig, ax = plt.subplots(2, 2, figsize=(12.6, 8.4))

    # ---------------------------------------------------------------- Panel 1 (0, 0): Distribution
    a = ax[0, 0]
    # Using explicit bins to reveal the bimodal distribution in systolic blood pressure
    bin_width = 2
    bins = np.arange(df["systolic"].min(), df["systolic"].max() + bin_width, bin_width)
    a.hist(df["systolic"], bins=bins, color=PALETTE["series"][0], edgecolor="white", linewidth=0.5)
    a.set_title("Systolic blood pressure shows a bimodal clinical distribution")
    a.set_xlabel("Systolic Blood Pressure (mmHg)")
    a.set_ylabel("Patient Count")
    a.text(0.70, 0.85, f"binwidth = {bin_width}", transform=a.transAxes, fontsize=9, color=PALETTE["ink_soft"])

   # ---------------------------------------------------------------- Panel 2 (0, 1): Box Plot
    b = ax[0, 1]
    # Sort wards by their median length of stay (los_days) rather than alphabetically
    ward_order = df.groupby("ward_clean")["los_days"].median().sort_values(ascending=False).index
    
    # Fix for Seaborn v0.13/v0.14: Assign hue to x and disable legend to avoid warnings
    num_wards = len(ward_order)
    box_palette = [PALETTE["series"][1]] * num_wards
    
    sns.boxplot(data=df, x="ward_clean", y="los_days", order=ward_order, ax=b,
                hue="ward_clean", palette=box_palette, legend=False, fliersize=2)
    b.set_title("Length of stay varies significantly across hospital wards")
    b.set_xlabel("Ward")
    # Using a log scale because length of stay is right-skewed
    b.set_yscale("log")
    b.set_ylabel("Length of Stay (Days, Log Scale)")

    # ---------------------------------------------------------------- Panel 3 (1, 0): Scatter Plot
    c = ax[1, 0]
    # Scatter plot exploring the relationship between BMI and Cholesterol, using transparency for 10k points
    c.scatter(df["bmi"], df["chol_mmol_l"], alpha=0.15, s=10, color=PALETTE["series"][2], edgecolors="none")
    c.set_title("Body Mass Index shows positive correlation with cholesterol levels")
    c.set_xlabel("BMI ($kg/m^2$)")
    c.set_chol = c.set_ylabel("Cholesterol (mmol/L)")

    # ---------------------------------------------------------------- Panel 4 (1, 1): Heatmap
    d = ax[1, 1]
    # Exclude 'map_mmhg' as a derived column (arithmetically dependent on systolic/diastolic)
    cols = ["age", "systolic", "diastolic", "heart_rate",
            "spo2", "bmi", "chol_mmol_l", "glucose_mmol_l", "los_days"]
    corr_matrix = df[cols].corr()
    
    # Mask the upper triangle since it duplicates information
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(corr_matrix, mask=mask, ax=d, cmap="RdBu_r", vmin=-1, vmax=1, center=0,
                annot=True, fmt=".2f", cbar_kws={"shrink": 0.8}, annot_kws={"size": 8})
    d.set_title("Clinical correlation screening matrix (excluding derived MAP)")

    # ---------------------------------------------------------------- Finishing Touches
    # Overall super title stating the primary finding
    fig.suptitle("Clinical Review Analysis: Patient Vitals, Ward Stay Distributions, and Risk Indicators", 
                 fontsize=14, fontweight="bold", x=0.12, y=0.98, ha="left")

    # Remove top and right spines for a clean, professional look on every panel
    for p in ax.flat:
        p.spines["top"].set_visible(False)
        p.spines["right"].set_visible(False)

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    return fig

fig = build_figure(df)

# Construct the required detailed source note
note = f"n = {len(df):,} · Invalid BMI and heart rate records removed · binwidth = 2 (Systolic) · Synthetic clinical dataset"

out = save_fig(fig, os.path.join(ROOT, "partD_figure.png"), note)
plt.close(fig)
print("\nWrote figure to:", out)

print("\nYour cleaning log — Part E item 3:\n")
for k, v in LOG.items():
    print(f"  {k:<36} {v}")


Wrote figure to: .\partD_figure.png

Your cleaning log — Part E item 3:

  rows loaded                          10,000
  ward spellings conformed             Conformed to 8 unique wards
  dates parsed                         Parsed 6,635 valid datetime entries
  bmi sentinels removed                Dropped 486 invalid BMI records (mean shifted from 66.18 to 27.35)
  heart-rate device faults removed     Dropped 37 rows with invalid heart rates outside [30, 250]
  derived column excluded              Excluded map_mmhg from correlation matrix to prevent artificial collinearity
